## Part 09. 프로젝트 루트 확인과 4개 CSV 불러오기


35. 프로젝트 루트 설정
[출처] Chapter 04. pandas로 데이터에 질문하기|작성자 아토믹데브

In [1]:

from pathlib import Path
project_root = Path.cwd()

if project_root.name == "notebooks":

    project_root = project_root.parent

data_dir = project_root / "data" / "raw"

print("프로젝트 루트:", project_root)

print("데이터 폴더:", data_dir)
print("데이터 폴더 존재:", data_dir.exists())

프로젝트 루트: c:\dev\ai-data-analysis
데이터 폴더: c:\dev\ai-data-analysis\data\raw
데이터 폴더 존재: True


## 36. pandas와 CSV 불러오기

In [2]:

import pandas as pd

customers = pd.read_csv(data_dir / "customers.csv")
products = pd.read_csv(data_dir / "products.csv")
orders = pd.read_csv(data_dir / "orders.csv")
order_items = pd.read_csv(data_dir / "order_items.csv")


## 37. 기본 구조와 주요 키 확인

In [3]:

datasets = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
}

for name, df in datasets.items():
    print(name, df.shape, df.columns.tolist())

customers (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
products (100, 4) ['product_id', 'product_name', 'category', 'price']
orders (301, 5) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
order_items (764, 5) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']


In [4]:
key_checks = {

    "customers.customer_id": customers["customer_id"],
    "products.product_id": products["product_id"],
    "orders.order_id": orders["order_id"],
    "order_items.order_item_id": order_items["order_item_id"],
}

for name, series in key_checks.items():

    print(
        name,
        "결측:", series.isna().sum(),
        "중복:", series.duplicated().sum(),
    )

customers.customer_id 결측: 0 중복: 0
products.product_id 결측: 0 중복: 0
orders.order_id 결측: 0 중복: 0
order_items.order_item_id 결측: 0 중복: 0


## Part 10. 컬럼 선택·조건 필터링·정렬

## 38. Series와 DataFrame 선택

In [5]:
city_series = customers["city"]
customer_view = customers[
    ["customer_id", "gender", "age", "city"]
]

print(type(city_series))
print(type(customer_view))
display(customer_view.head())

<class 'pandas.Series'>
<class 'pandas.DataFrame'>


,customer_id,gender,age,city
0,1,F,19,광주
1,2,F,32,대구
2,3,F,61,성남
3,4,F,55,울산
4,5,F,19,부산


## 39. 단일 조건 필터링

In [6]:

customers_over_30 = customers[
    customers["age"] >= 30
]

print(len(customers), len(customers_over_30))
display(customers_over_30.head())

150 111


,customer_id,name,gender,age,city,signup_date
1,2,김정호,F,32,대구,2025-11-30
2,3,이경수,F,61,성남,2024-07-10
3,4,조영호,F,55,울산,2026-05-11
5,6,김지원,F,32,성남,2026-07-25
6,7,이상현,F,53,인천,2025-01-09


## 40. 복합 조건 필터링

# 30세 이상이면서 서울 거주:

In [7]:
seoul_over_30 = customers[
    (customers["age"] >= 30)
    & (customers["city"] == "서울")
]

In [8]:

seoul_or_busan = customers[
    customers["city"].isin(["서울", "부산"])
]
display(
    seoul_or_busan["city"].value_counts()
)

city
부산    16
서울    15
Name: count, dtype: int64

In [9]:
not_completed = orders[
    ~(orders["order_status"] == "completed")
]
display(

    not_completed["order_status"].value_counts(
        dropna=False
    )

)

order_status
cancelled    65
refunded     52
Name: count, dtype: int64

## 41. 상품 가격 정렬

In [10]:
expensive_products = (
    products
    .sort_values("price", ascending=False)
    .head(10)
)

display(
    expensive_products[
        [
            "product_id",
            "product_name",
            "category",
            "price",
        ]
    ]
)

,product_id,product_name,category,price
98,99,뷰티 상품 099,뷰티,200000
69,70,패션 상품 070,패션,198000
57,58,식품 상품 058,식품,197000
42,43,뷰티 상품 043,뷰티,197000
23,24,스포츠 상품 024,스포츠,196000
8,9,스포츠 상품 009,스포츠,193000
36,37,뷰티 상품 037,뷰티,193000
71,72,뷰티 상품 072,뷰티,189000
7,8,스포츠 상품 008,스포츠,189000
52,53,생활용품 상품 053,생활용품,188000


## Part 11. line_total 생성과 전체 주문 금액 구분

## 42. 작업용 복사본과 파생 컬럼

In [11]:
order_items_work = order_items.copy()
order_items_work["line_total"] = (
    order_items_work["quantity"]
    * order_items_work["unit_price"]

)

In [12]:
display(

    order_items_work[

        [

            "order_item_id",

            "order_id",

            "product_id",

            "quantity",

            "unit_price",

            "line_total",

        ]

    ].head()

)

,order_item_id,order_id,product_id,quantity,unit_price,line_total
0,1,1,100,3,102000,306000
1,2,1,87,5,25000,125000
2,3,1,7,3,142000,426000
3,4,1,9,3,193000,579000
4,5,2,72,4,189000,756000


## 43. 수작업 검증

In [13]:

sample = order_items_work.iloc[0]
expected = sample["quantity"] * sample["unit_price"]
actual = sample["line_total"]
print("수작업:", expected)
print("파생 컬럼:", actual)
print("일치:", expected == actual)

수작업: 306000
파생 컬럼: 306000
일치: True



## 44. 전체 주문상세 금액

In [15]:

all_order_amount = order_items_work["line_total"].sum()
print("전체 주문상세 금액:", all_order_amount)

전체 주문상세 금액: 255610000



현재 합계에는 취소 또는 환불 주문이 포함될 수 있으므로
완료 주문 기준 매출이 아니라 전체 주문상세 금액으로 표현한다.

## Part 12. 주문 데이터 병합과 완료 주문 분석셋 만들기

# 45. 병합용 주문 컬럼 선택

In [16]:
orders_for_merge = orders[
    [
        "order_id",
        "customer_id",
        "order_date",
        "order_status",
    ]
].copy()

## 46. 주문상세와 주문 병합

In [17]:

order_sales = (
    order_items_work
    .merge(
        orders_for_merge,
        on="order_id",
        how="left",
        validate="many_to_one",
        indicator="order_match",
    )
)

## 47. 병합 검증

In [18]:

print("병합 전 행 수:", len(order_items_work))
print("병합 후 행 수:", len(order_sales))
display(
    order_sales["order_match"].value_counts(
        dropna=False
    )
)

병합 전 행 수: 764
병합 후 행 수: 764


order_match
both          764
left_only       0
right_only      0
Name: count, dtype: int64

In [19]:
unmatched_orders = order_sales[
    order_sales["order_match"] != "both"
]
display(unmatched_orders.head())

,order_item_id,order_id,product_id,quantity,unit_price,line_total,customer_id,order_date,order_status,order_match


## 48. 완료 주문 분석셋

In [20]:

display(
    order_sales["order_status"].value_counts(
        dropna=False
    )
)

order_status
completed    474
cancelled    162
refunded     128
Name: count, dtype: int64

In [21]:
completed_sales = order_sales[
    order_sales["order_status"] == "completed"
].copy()

In [22]:
print("완료 주문상세 행:", len(completed_sales))
print(
    "완료 주문 수:",
    completed_sales["order_id"].nunique(),
)

print(
    "완료 주문 고객 수:",
    completed_sales["customer_id"].nunique(),
)

print(
    "완료 주문 매출:",
    completed_sales["line_total"].sum(),
)

완료 주문상세 행: 474
완료 주문 수: 184
완료 주문 고객 수: 100
완료 주문 매출: 148990000


## Part 13. 상품 데이터 병합과 카테고리·상품 매출

## 49. 필요한 상품 정보만 선택

In [23]:

products_for_merge = products[
    [
        "product_id",
        "product_name",
        "category",
    ]
].copy()

## 50. 완료 주문상세와 상품 병합

In [31]:

completed_items = (
    completed_sales
    .merge(
        products_for_merge,
        on="product_id",
        how="left",
        validate="many_to_one",
        indicator="product_match",
    )
)

In [25]:

print(len(completed_sales), len(completed_items))
display(
    completed_items["product_match"].value_counts(
        dropna=False
    )
)

474 474


product_match
both          474
left_only       0
right_only      0
Name: count, dtype: int64

### 51. 카테고리별 매출

In [30]:

category_sales = (
    completed_items
    .groupby("category", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
        quantity_sold=("quantity", "sum"),
        detail_row_count=("order_item_id", "count"),
    )
    .sort_values("total_sales", ascending=False)
)
display(category_sales)

,category,total_sales,order_count,customer_count,quantity_sold,detail_row_count
3,스포츠,31743000,85,67,295,100
5,전자기기,26400000,60,44,259,78
2,생활용품,23915000,65,50,272,83
1,뷰티,23383000,65,53,223,76
4,식품,16573000,36,31,133,42
0,도서,16389000,52,46,149,58
6,패션,10587000,33,27,111,37


## 52. 카테고리 합계 검증

In [29]:
category_total = category_sales["total_sales"].sum()
completed_total = completed_items["line_total"].sum()
print(category_total)
print(completed_total)
print(category_total == completed_total)

148990000
148990000
True


## 53. 상품별 매출

In [28]:
product_sales = (
    completed_items
    .groupby(
        ["product_id", "product_name", "category"],
        as_index=False,
    )
    .agg(
        total_sales=("line_total", "sum"),
        quantity_sold=("quantity", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
    )
    .sort_values("total_sales", ascending=False)
)
display(product_sales.head(10))

,product_id,product_name,category,total_sales,quantity_sold,order_count,customer_count
39,41,스포츠 상품 041,스포츠,5705000,35,12,11
11,12,식품 상품 012,식품,4375000,25,7,7
8,9,스포츠 상품 009,스포츠,3860000,20,6,5
70,72,뷰티 상품 072,뷰티,3780000,20,6,6
69,71,전자기기 상품 071,전자기기,3703000,23,5,5
66,68,스포츠 상품 068,스포츠,3640000,26,8,8
78,81,전자기기 상품 081,전자기기,3630000,22,6,6
10,11,패션 상품 011,패션,3565000,31,7,7
20,22,생활용품 상품 022,생활용품,3248000,29,8,8
86,89,생활용품 상품 089,생활용품,3090000,30,11,11


## Part 14. 월별 매출과 고객별 구매 금액

### 54. 주문 날짜 변환과 주문 월 생성

In [32]:
completed_items["order_date"] = pd.to_datetime(
    completed_items["order_date"],
    errors="coerce",
)
print(
    "날짜 변환 실패:",
    completed_items["order_date"].isna().sum(),
)

날짜 변환 실패: 0


In [33]:

completed_items["order_month"] = (
    completed_items["order_date"]
    .dt.to_period("M")
    .astype("string")
)

## 55. 월별 매출

In [34]:

monthly_sales = (
    completed_items
    .groupby("order_month", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        customer_count=("customer_id", "nunique"),
        quantity_sold=("quantity", "sum"),
    )
    .sort_values("order_month")
)
display(monthly_sales)

,order_month,total_sales,order_count,customer_count,quantity_sold
0,2025-08,5869000,8,8,52
1,2025-09,16147000,19,19,132
2,2025-10,12385000,15,15,120
3,2025-11,23550000,24,23,233
4,2025-12,9876000,13,13,99
5,2026-01,10851000,13,13,105
6,2026-02,16504000,21,20,150
7,2026-03,9885000,18,16,102
8,2026-04,15536000,17,16,157
9,2026-05,15310000,19,18,152


## 56. 고객별 구매 금액

In [35]:

customer_sales = (
    completed_items
    .groupby("customer_id", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
        quantity_sold=("quantity", "sum"),
    )
)